# Praktikum Evaluasi Model Komprehensif pada Dataset Imbalanced

**Studi kasus:** Credit Card Fraud Detection  
**Cakupan:** poin A–P dari modul (poin Q/tugas praktikum tidak dikerjakan).

Tujuan utama adalah mendeteksi fraud dengan *recall* tinggi tanpa mengorbankan *precision* secara berlebihan. Selain ROC-AUC, notebook melaporkan PR-AUC karena lebih informatif untuk kelas positif yang sangat langka.

## A–B. Studi kasus dan pemuatan dataset

Dataset ULB berisi 284.807 transaksi dengan 492 fraud. Jika file belum tersedia, sel berikut mengunduhnya dari penyimpanan publik TensorFlow.

In [1]:
from pathlib import Path
from urllib.request import urlretrieve

DATA = Path('data/creditcard.csv')
DATA.parent.mkdir(exist_ok=True)
if not DATA.exists():
    url = 'https://storage.googleapis.com/download.tensorflow.org/data/creditcard.csv'
    print('Mengunduh dataset (sekitar 144 MB)...')
    urlretrieve(url, DATA)
print(f'Dataset: {DATA} ({DATA.stat().st_size / 1024**2:.1f} MB)')

Dataset: data/creditcard.csv (143.8 MB)


## C. Import library dan setup

In [2]:
from pathlib import Path
import json
import warnings

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, average_precision_score, confusion_matrix, f1_score,
    precision_recall_curve, precision_score, recall_score, roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import (
    GridSearchCV, StratifiedKFold, cross_validate, train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

warnings.filterwarnings("ignore")
np.random.seed(42)

ROOT = Path.cwd()
DATA = ROOT / "data" / "creditcard.csv"
OUT = ROOT / "outputs"
FIG = ROOT / "figures"
OUT.mkdir(exist_ok=True)
FIG.mkdir(exist_ok=True)


def metrics(y_true, y_pred, y_prob):
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "ROC_AUC": roc_auc_score(y_true, y_prob),
        "PR_AUC": average_precision_score(y_true, y_prob),
        "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
    }


def save_cm(y_true, y_pred, title, filename):
    plt.figure(figsize=(6, 5))
    sns.heatmap(confusion_matrix(y_true, y_pred), annot=True, fmt="d", cmap="Blues",
                xticklabels=["Normal", "Fraud"], yticklabels=["Normal", "Fraud"])
    plt.title(title); plt.xlabel("Predicted"); plt.ylabel("Actual")
    plt.tight_layout(); plt.savefig(FIG / filename, dpi=150); plt.show()

## D. Load dataset dan eksplorasi awal

In [3]:
print("C-D. Load dataset dan eksplorasi")
df = pd.read_csv(DATA)
print("Shape:", df.shape)
print("Missing values:", int(df.isna().sum().sum()))
print("Distribusi kelas:\n", df["Class"].value_counts())
print(f"Proporsi fraud: {df['Class'].mean()*100:.4f}%")
print("Statistik Amount:\n", df["Amount"].describe())
print("Mean Amount fraud:", df.loc[df.Class == 1, "Amount"].mean())
print("Mean Amount normal:", df.loc[df.Class == 0, "Amount"].mean())

counts = df["Class"].value_counts().sort_index()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(["Normal", "Fraud"], counts.values, color=["steelblue", "coral"])
axes[0].set_title("Distribusi Kelas"); axes[0].set_ylabel("Jumlah")
axes[1].pie(counts.values, labels=["Normal", "Fraud"], autopct="%1.3f%%",
            colors=["steelblue", "coral"])
axes[1].set_title("Proporsi Kelas")
plt.tight_layout(); plt.savefig(FIG / "01_distribusi_kelas.png", dpi=150); plt.show()

C-D. Load dataset dan eksplorasi


Shape: (284807, 31)
Missing values: 0
Distribusi kelas:
 Class
0    284315
1       492
Name: count, dtype: int64
Proporsi fraud: 0.1727%
Statistik Amount:
 count    284807.000000
mean         88.349619
std         250.120109
min           0.000000
25%           5.600000
50%          22.000000
75%          77.165000
max       25691.160000
Name: Amount, dtype: float64
Mean Amount fraud: 122.21132113821139
Mean Amount normal: 88.29102242231328


**Temuan awal:** akurasi berpotensi menipu karena fraud hanya sekitar 0,172%. Dataset tidak memiliki missing value.

## E. Preprocessing dan split data

In [4]:
print("\nE. Split data (scaling sengaja belum dilakukan agar tidak leakage)")
X = df.drop(columns="Class")
y = df["Class"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Train:", X_train.shape, np.bincount(y_train))
print("Test :", X_test.shape, np.bincount(y_test))

all_results = []


E. Split data (scaling sengaja belum dilakukan agar tidak leakage)


Train: (227845, 30) [227451    394]
Test : (56962, 30) [56864    98]


Scaling belum dilakukan pada seluruh data. `StandardScaler` ditempatkan dalam pipeline agar hanya mempelajari statistik training set/fold dan mencegah leakage.

## F. Naive baseline dan akurasi yang menipu

In [5]:
print("\nF. Naive baseline")
y_pred_naive = np.zeros(len(y_test), dtype=int)
naive = metrics(y_test, y_pred_naive, y_pred_naive.astype(float))
all_results.append({"Strategi": "Naive Baseline", **naive})
print(naive)


F. Naive baseline
{'Accuracy': 0.9982795547909132, 'Precision': 0.0, 'Recall': 0.0, 'F1': 0.0, 'ROC_AUC': np.float64(0.5), 'PR_AUC': np.float64(0.0017204452090867595), 'TN': 56864, 'FP': 0, 'FN': 98, 'TP': 0}


Model yang selalu memilih kelas normal memperoleh akurasi hampir 100%, tetapi recall fraud = 0. Artinya seluruh transaksi fraud lolos.

## G–H. Evaluasi Logistic Regression dan confusion matrix

In [6]:
print("\nG-H. Logistic Regression baseline")
lr_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=2000, random_state=42)),
])
lr_pipe.fit(X_train, y_train)
y_pred_lr = lr_pipe.predict(X_test)
y_prob_lr = lr_pipe.predict_proba(X_test)[:, 1]
lr_result = metrics(y_test, y_pred_lr, y_prob_lr)
all_results.append({"Strategi": "LR Baseline", **lr_result})
print(lr_result)
save_cm(y_test, y_pred_lr, "Confusion Matrix – Logistic Regression", "02_cm_lr.png")


G-H. Logistic Regression baseline


{'Accuracy': 0.9991397773954567, 'Precision': 0.8266666666666667, 'Recall': 0.6326530612244898, 'F1': 0.7167630057803468, 'ROC_AUC': np.float64(0.9605494455801453), 'PR_AUC': np.float64(0.7413820992780461), 'TN': 56851, 'FP': 13, 'FN': 36, 'TP': 62}


## I. Handling imbalance dengan SMOTE

In [7]:
print("\nI. Logistic Regression + SMOTE (hanya training)")
smote_pipe = ImbPipeline([
    ("scaler", StandardScaler()),
    ("smote", SMOTE(random_state=42)),
    ("classifier", LogisticRegression(max_iter=2000, random_state=42)),
])
smote_pipe.fit(X_train, y_train)
y_pred_smote = smote_pipe.predict(X_test)
y_prob_smote = smote_pipe.predict_proba(X_test)[:, 1]
smote_result = metrics(y_test, y_pred_smote, y_prob_smote)
all_results.append({"Strategi": "LR + SMOTE", **smote_result})
print(smote_result)
save_cm(y_test, y_pred_smote, "Confusion Matrix – LR + SMOTE", "03_cm_smote.png")


I. Logistic Regression + SMOTE (hanya training)


{'Accuracy': 0.9741055440469084, 'Precision': 0.057803468208092484, 'Recall': 0.9183673469387755, 'F1': 0.10876132930513595, 'ROC_AUC': np.float64(0.9708434302252134), 'PR_AUC': np.float64(0.7244694356694709), 'TN': 55397, 'FP': 1467, 'FN': 8, 'TP': 90}


SMOTE hanya dijalankan pada training set melalui `imblearn.Pipeline`. Test set tetap merepresentasikan distribusi data nyata.

## J. Stratified 5-Fold Cross-Validation

In [8]:
print("\nJ. Stratified 5-fold cross-validation")
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ["accuracy", "precision", "recall", "f1", "roc_auc", "average_precision"]
cv_rows = []
cv_models = {
    "Logistic Regression": lr_pipe,
}
for name, model in cv_models.items():
    scores = cross_validate(model, X_train, y_train, cv=skf, scoring=scoring, n_jobs=4)
    row = {"Model": name}
    for key in scoring:
        vals = scores[f"test_{key}"]
        row[f"{key}_mean"] = vals.mean()
        row[f"{key}_std"] = vals.std(ddof=1)
    cv_rows.append(row)
    print(name, row)

# Random Forest juga menggunakan sampel stratified agar notebook cepat direproduksi.
X_rf, _, y_rf, _ = train_test_split(
    X_train, y_train, train_size=40000, random_state=42, stratify=y_train
)
rf_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", RandomForestClassifier(n_estimators=30, random_state=42, n_jobs=1,
                                          class_weight="balanced_subsample")),
])
rf_scores = cross_validate(rf_pipe, X_rf, y_rf, cv=skf, scoring=scoring, n_jobs=4)
rf_row = {"Model": "Random Forest (sampel stratified 40k)"}
for key in scoring:
    vals = rf_scores[f"test_{key}"]
    rf_row[f"{key}_mean"] = vals.mean()
    rf_row[f"{key}_std"] = vals.std(ddof=1)
cv_rows.append(rf_row)
print(rf_row)

# RBF-SVM pada sampel stratified karena kompleksitas kernel kuadratik tidak realistis
# untuk keseluruhan 227 ribu baris training.
X_svm, _, y_svm, _ = train_test_split(
    X_train, y_train, train_size=10000, random_state=42, stratify=y_train
)
svm_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", SVC(kernel="rbf", probability=False, random_state=42)),
])
svm_scores = cross_validate(svm_pipe, X_svm, y_svm, cv=skf, scoring=scoring, n_jobs=4)
svm_row = {"Model": "SVM RBF (sampel stratified 10k)"}
for key in scoring:
    vals = svm_scores[f"test_{key}"]
    svm_row[f"{key}_mean"] = vals.mean()
    svm_row[f"{key}_std"] = vals.std(ddof=1)
cv_rows.append(svm_row)
print(svm_row)
pd.DataFrame(cv_rows).to_csv(OUT / "cv_models.csv", index=False)



J. Stratified 5-fold cross-validation


Logistic Regression {'Model': 'Logistic Regression', 'accuracy_mean': np.float64(0.9992012113498211), 'accuracy_std': np.float64(4.807852333866309e-05), 'precision_mean': np.float64(0.8760455835737655), 'precision_std': np.float64(0.043993243963736925), 'recall_mean': np.float64(0.6294060370009736), 'recall_std': np.float64(0.024944842060743995), 'f1_mean': np.float64(0.7315586528995093), 'f1_std': np.float64(0.012650636899151916), 'roc_auc_mean': np.float64(0.9783458160812424), 'roc_auc_std': np.float64(0.011022620467945998), 'average_precision_mean': np.float64(0.7614969310778783), 'average_precision_std': np.float64(0.045965510443186926)}


{'Model': 'Random Forest (sampel stratified 40k)', 'accuracy_mean': np.float64(0.9993000000000001), 'accuracy_std': np.float64(0.0003491060010942308), 'precision_mean': np.float64(0.8984848484848484), 'precision_std': np.float64(0.10841541709089889), 'recall_mean': np.float64(0.6626373626373626), 'recall_std': np.float64(0.1663211408495358), 'f1_mean': np.float64(0.7557246963562753), 'f1_std': np.float64(0.14958205746661876), 'roc_auc_mean': np.float64(0.9395425111342351), 'roc_auc_std': np.float64(0.07599032970002997), 'average_precision_mean': np.float64(0.7906971067681152), 'average_precision_std': np.float64(0.1847281944336079)}


/Users/ar/Project/MachineLearning/.venv/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/ar/Project/MachineLearning/.venv/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/ar/Project/MachineLearning/.venv/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", le

/Users/ar/Project/MachineLearning/.venv/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


{'Model': 'SVM RBF (sampel stratified 10k)', 'accuracy_mean': np.float64(0.9983000000000001), 'accuracy_std': np.float64(0.0002738612787526137), 'precision_mean': np.float64(0.0), 'precision_std': np.float64(0.0), 'recall_mean': np.float64(0.0), 'recall_std': np.float64(0.0), 'f1_mean': np.float64(0.0), 'f1_std': np.float64(0.0), 'roc_auc_mean': np.float64(0.9465422206455976), 'roc_auc_std': np.float64(0.10251718705162884), 'average_precision_mean': np.float64(0.7526504004470106), 'average_precision_std': np.float64(0.2066630127144225)}


RBF-SVM dievaluasi pada sampel stratified 10.000 baris karena kompleksitas kernel pada 227 ribu data training tidak praktis untuk notebook pembelajaran.

## K. Implementasi manual Stratified K-Fold

In [9]:
print("\nK. Manual Stratified 5-fold")
manual_rows = []
for fold, (tr, va) in enumerate(skf.split(X_train, y_train), start=1):
    model = clone(lr_pipe)
    model.fit(X_train.iloc[tr], y_train.iloc[tr])
    pred = model.predict(X_train.iloc[va])
    prob = model.predict_proba(X_train.iloc[va])[:, 1]
    row = {"Fold": fold, **metrics(y_train.iloc[va], pred, prob)}
    manual_rows.append(row)
    print(row)
manual_df = pd.DataFrame(manual_rows)
manual_df.to_csv(OUT / "manual_cv.csv", index=False)
print("Mean ± sample std:\n", manual_df[["Accuracy", "F1", "Recall", "ROC_AUC"]].agg(["mean", "std"]))



K. Manual Stratified 5-fold


{'Fold': 1, 'Accuracy': 0.9992319339902126, 'Precision': 0.9056603773584906, 'Recall': 0.6153846153846154, 'F1': 0.732824427480916, 'ROC_AUC': np.float64(0.9830158008149259), 'PR_AUC': np.float64(0.8196816824053996), 'TN': 45486, 'FP': 5, 'FN': 30, 'TP': 48}


{'Fold': 2, 'Accuracy': 0.9992319339902126, 'Precision': 0.9230769230769231, 'Recall': 0.6075949367088608, 'F1': 0.732824427480916, 'ROC_AUC': np.float64(0.9726371910922139), 'PR_AUC': np.float64(0.7566834216049007), 'TN': 45486, 'FP': 4, 'FN': 31, 'TP': 48}


{'Fold': 3, 'Accuracy': 0.999188044503939, 'Precision': 0.828125, 'Recall': 0.6708860759493671, 'F1': 0.7412587412587412, 'ROC_AUC': np.float64(0.9648563740535547), 'PR_AUC': np.float64(0.7032690892178198), 'TN': 45479, 'FP': 11, 'FN': 26, 'TP': 53}


{'Fold': 4, 'Accuracy': 0.9991222102745287, 'Precision': 0.8305084745762712, 'Recall': 0.620253164556962, 'F1': 0.7101449275362319, 'ROC_AUC': np.float64(0.9940980212649324), 'PR_AUC': np.float64(0.7925643877654285), 'TN': 45480, 'FP': 10, 'FN': 30, 'TP': 49}


{'Fold': 5, 'Accuracy': 0.9992319339902126, 'Precision': 0.8928571428571429, 'Recall': 0.6329113924050633, 'F1': 0.7407407407407407, 'ROC_AUC': np.float64(0.977121693180585), 'PR_AUC': np.float64(0.7352860743958433), 'TN': 45484, 'FP': 6, 'FN': 29, 'TP': 50}
Mean ± sample std:
       Accuracy        F1    Recall   ROC_AUC
mean  0.999201  0.731559  0.629406  0.978346
std   0.000048  0.012651  0.024945  0.011023


## L. GridSearchCV dengan pipeline

In [10]:
print("\nL. GridSearchCV")
# Validation terpisah juga dipakai pada bagian N agar threshold tidak dipilih di test set.
X_core, X_val, y_core, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
)
param_grid = {
    "classifier__solver": ["liblinear"],
    "classifier__penalty": ["l1", "l2"],
    "classifier__C": [0.01, 0.1, 1.0, 10.0],
}
grid = GridSearchCV(lr_pipe, param_grid, cv=skf, scoring="f1", n_jobs=4,
                    return_train_score=False, verbose=1)
grid.fit(X_core, y_core)
print("Best params:", grid.best_params_)
print("Best CV F1:", grid.best_score_)
pd.DataFrame(grid.cv_results_).sort_values("rank_test_score").to_csv(
    OUT / "grid_search_results.csv", index=False
)

y_pred_best = grid.best_estimator_.predict(X_test)
y_prob_best = grid.best_estimator_.predict_proba(X_test)[:, 1]
best_result = metrics(y_test, y_pred_best, y_prob_best)
all_results.append({"Strategi": "Best GridSearch (threshold 0.5)", **best_result})
print("Test metrics:", best_result)



L. GridSearchCV


Fitting 5 folds for each of 8 candidates, totalling 40 fits


Best params: {'classifier__C': 1.0, 'classifier__penalty': 'l2', 'classifier__solver': 'liblinear'}
Best CV F1: 0.7642002969141968
Test metrics: {'Accuracy': 0.9991573329588147, 'Precision': 0.8289473684210527, 'Recall': 0.6428571428571429, 'F1': 0.7241379310344828, 'ROC_AUC': np.float64(0.9480033994464415), 'PR_AUC': np.float64(0.7440527141109519), 'TN': 56851, 'FP': 13, 'FN': 35, 'TP': 63}


## M. ROC Curve dan AUC

In [11]:
print("\nM. ROC Curve dan AUC")
fpr, tpr, _ = roc_curve(y_test, y_prob_best)
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f"ROC-AUC = {best_result['ROC_AUC']:.4f}")
plt.plot([0, 1], [0, 1], "--", label="Random = 0.5")
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title("ROC Curve – Model GridSearch"); plt.legend(); plt.grid(alpha=.3)
plt.tight_layout(); plt.savefig(FIG / "04_roc_curve.png", dpi=150); plt.show()



M. ROC Curve dan AUC


## N. Threshold tuning

In [12]:
print("\nN. Threshold tuning pada validation set (bukan test set)")
val_prob = grid.best_estimator_.predict_proba(X_val)[:, 1]
prec, rec, th = precision_recall_curve(y_val, val_prob)
f1s = 2 * prec * rec / (prec + rec + 1e-12)
valid_f1s = f1s[:-1]
idx = int(np.nanargmax(valid_f1s))
optimal_threshold = float(th[idx])
print("Optimal threshold:", optimal_threshold, "Validation F1:", valid_f1s[idx])
plt.figure(figsize=(9, 6))
plt.plot(th, prec[:-1], label="Precision")
plt.plot(th, rec[:-1], label="Recall")
plt.plot(th, valid_f1s, label="F1")
plt.axvline(optimal_threshold, color="black", ls="--", label=f"Optimal={optimal_threshold:.3f}")
plt.xlabel("Threshold"); plt.ylabel("Score"); plt.title("Threshold Tuning pada Validation Set")
plt.legend(); plt.grid(alpha=.3); plt.tight_layout()
plt.savefig(FIG / "05_threshold_tuning.png", dpi=150); plt.show()
y_pred_opt = (y_prob_best >= optimal_threshold).astype(int)
opt_result = metrics(y_test, y_pred_opt, y_prob_best)
all_results.append({"Strategi": f"Threshold Tuning ({optimal_threshold:.4f})", **opt_result})
print("Test metrics threshold optimal:", opt_result)



N. Threshold tuning pada validation set (bukan test set)
Optimal threshold: 0.05760576106400541 Validation F1: 0.7682119205293023


Test metrics threshold optimal: {'Accuracy': 0.9989993328885924, 'Precision': 0.6782608695652174, 'Recall': 0.7959183673469388, 'F1': 0.7323943661971831, 'ROC_AUC': np.float64(0.9480033994464415), 'PR_AUC': np.float64(0.7440527141109519), 'TN': 56827, 'FP': 37, 'FN': 20, 'TP': 78}


Threshold dipilih pada validation set, bukan test set. Dengan demikian test set tetap menjadi evaluasi akhir yang independen.

## O. Demonstrasi data leakage

In [13]:
print("\nO. Demonstrasi data leakage")
# Leakage sederhana mungkin hanya menghasilkan selisih kecil pada dataset besar ini.
wrong_scaler = StandardScaler()
X_wrong = wrong_scaler.fit_transform(X)
Xtr_w, Xte_w, ytr_w, yte_w = train_test_split(
    X_wrong, y, test_size=.2, random_state=42, stratify=y
)
wrong_model = LogisticRegression(max_iter=2000, random_state=42).fit(Xtr_w, ytr_w)
wrong_prob = wrong_model.predict_proba(Xte_w)[:, 1]
wrong_metrics = metrics(yte_w, wrong_model.predict(Xte_w), wrong_prob)
right_metrics = lr_result
leakage = pd.DataFrame([
    {"Cara": "SALAH: scaler fit seluruh data", **wrong_metrics},
    {"Cara": "BENAR: scaler dalam pipeline", **right_metrics},
])
print(leakage[["Cara", "Accuracy", "Precision", "Recall", "F1", "ROC_AUC"]])
leakage.to_csv(OUT / "leakage_comparison.csv", index=False)



O. Demonstrasi data leakage


                             Cara  Accuracy  Precision    Recall        F1  \
0  SALAH: scaler fit seluruh data  0.999157   0.828947  0.642857  0.724138   
1    BENAR: scaler dalam pipeline  0.999140   0.826667  0.632653  0.716763   

    ROC_AUC  
0  0.959947  
1  0.960549  


Selisih dapat sangat kecil pada dataset besar, tetapi cara pertama tetap salah: mean dan standar deviasi test set ikut memengaruhi preprocessing training.

## P. Perbandingan final semua strategi

In [14]:
print("\nP. Perbandingan final")
comparison = pd.DataFrame(all_results)
comparison.to_csv(OUT / "final_comparison.csv", index=False)
print(comparison.to_string(index=False))
plot_cols = ["Accuracy", "Precision", "Recall", "F1", "ROC_AUC", "PR_AUC"]
comparison.set_index("Strategi")[plot_cols].plot(kind="bar", figsize=(14, 7), colormap="Set2")
plt.title("Perbandingan Final Semua Strategi"); plt.ylabel("Score")
plt.xticks(rotation=25, ha="right"); plt.legend(loc="lower right", ncol=2)
plt.tight_layout(); plt.savefig(FIG / "06_final_comparison.png", dpi=150); plt.show()

summary = {
    "shape": list(df.shape),
    "fraud_count": int(y.sum()),
    "fraud_percent": float(y.mean() * 100),
    "best_params": grid.best_params_,
    "best_cv_f1": float(grid.best_score_),
    "optimal_threshold": optimal_threshold,
}
(OUT / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print("\nSelesai. Semua output tersimpan di outputs/ dan figures/.")


P. Perbandingan final
                       Strategi  Accuracy  Precision   Recall       F1  ROC_AUC   PR_AUC    TN   FP  FN  TP
                 Naive Baseline  0.998280   0.000000 0.000000 0.000000 0.500000 0.001720 56864    0  98   0
                    LR Baseline  0.999140   0.826667 0.632653 0.716763 0.960549 0.741382 56851   13  36  62
                     LR + SMOTE  0.974106   0.057803 0.918367 0.108761 0.970843 0.724469 55397 1467   8  90
Best GridSearch (threshold 0.5)  0.999157   0.828947 0.642857 0.724138 0.948003 0.744053 56851   13  35  63
      Threshold Tuning (0.0576)  0.998999   0.678261 0.795918 0.732394 0.948003 0.744053 56827   37  20  78



Selesai. Semua output tersimpan di outputs/ dan figures/.


## Kesimpulan

Pemilihan model tidak boleh didasarkan pada accuracy saja. Baseline naif gagal total mengenali fraud. SMOTE menaikkan recall tetapi dapat menghasilkan lebih banyak false positive. GridSearch meningkatkan konfigurasi model, sedangkan threshold tuning memberi kontrol langsung terhadap trade-off precision–recall. Strategi akhir harus mengikuti biaya bisnis FN dan FP.

# Q. Tugas Praktikum — Analisis Customer Churn

Bagian ini mengerjakan tugas praktikum menggunakan **IT Customer Churn Dataset** dari Kaggle. Seluruh langkah ditempatkan setelah praktikum A–P pada notebook yang sama.

Tujuan bisnisnya adalah mendeteksi pelanggan yang berpotensi berhenti berlangganan (`Churn = Yes`). Karena tindak lanjut retensi masih dapat diverifikasi oleh tim bisnis, **recall** menjadi prioritas agar pelanggan berisiko tidak banyak terlewat, tetapi precision tetap dipantau untuk mengendalikan biaya intervensi.

## Q.1 Persiapan data dan EDA

Dataset diunduh dari Kaggle melalui endpoint publik apabila belum tersedia secara lokal. File CSV mentah tidak disimpan di Git.

In [15]:
from urllib.request import urlretrieve
from zipfile import ZipFile

Q_DATA = Path("data/IT_customer_churn.csv")
Q_DATA.parent.mkdir(exist_ok=True)
if not Q_DATA.exists():
    q_zip = Q_DATA.parent / "it-customer-churn.zip"
    q_url = "https://www.kaggle.com/api/v1/datasets/download/soheiltehranipour/it-customer-churn"
    print("Mengunduh IT Customer Churn Dataset...")
    urlretrieve(q_url, q_zip)
    with ZipFile(q_zip) as zf:
        zf.extract("IT_customer_churn.csv", Q_DATA.parent)
    q_zip.unlink()
print(f"Dataset: {Q_DATA} ({Q_DATA.stat().st_size / 1024:.1f} KB)")

churn = pd.read_csv(Q_DATA)
# Spasi kosong pada TotalCharges menjadi NaN dan ditangani imputer di pipeline.
churn["TotalCharges"] = pd.to_numeric(churn["TotalCharges"], errors="coerce")
print("Shape:", churn.shape)
print("Duplikasi:", churn.duplicated().sum())
print("Missing values:")
print(churn.isna().sum()[churn.isna().sum() > 0])
display(churn.head())
display(churn.describe(include="all").T)

Dataset: data/IT_customer_churn.csv (878.9 KB)
Shape: (7043, 20)
Duplikasi: 22
Missing values:
TotalCharges    11
dtype: int64


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
gender,7043,2,Male,3555,NaN,NaN,NaN,NaN,NaN,NaN,NaN
SeniorCitizen,7043.0,NaN,NaN,NaN,0.162147,0.368612,0.0,0.0,0.0,0.0,1.0
Partner,7043,2,No,3641,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Dependents,7043,2,No,4933,NaN,NaN,NaN,NaN,NaN,NaN,NaN
tenure,7043.0,NaN,NaN,NaN,32.371149,24.559481,0.0,9.0,29.0,55.0,72.0
PhoneService,7043,2,Yes,6361,NaN,NaN,NaN,NaN,NaN,NaN,NaN
MultipleLines,7043,3,No,3390,NaN,NaN,NaN,NaN,NaN,NaN,NaN
InternetService,7043,3,Fiber optic,3096,NaN,NaN,NaN,NaN,NaN,NaN,NaN
OnlineSecurity,7043,3,No,3498,NaN,NaN,NaN,NaN,NaN,NaN,NaN
OnlineBackup,7043,3,No,3088,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [16]:
q_counts = churn["Churn"].value_counts()
print("Distribusi target:")
print(q_counts)
print(f"Proporsi churn: {(churn['Churn'].eq('Yes').mean() * 100):.2f}%")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(q_counts.index, q_counts.values, color=["steelblue", "coral"])
axes[0].set(title="Distribusi Customer Churn", xlabel="Churn", ylabel="Jumlah")
axes[1].pie(q_counts.values, labels=q_counts.index, autopct="%1.1f%%", colors=["steelblue", "coral"])
axes[1].set_title("Proporsi Target")
plt.tight_layout()
plt.savefig("figures/07_q_distribusi_churn.png", dpi=150, bbox_inches="tight")
plt.show()

Distribusi target:
Churn
No     5174
Yes    1869
Name: count, dtype: int64
Proporsi churn: 26.54%


**Temuan EDA:** kelas churn (`Yes`) mencakup sekitar 26,5% data sehingga terdapat ketidakseimbangan kelas. `TotalCharges` memiliki nilai kosong setelah konversi numerik. Penanganan missing value, scaling, dan encoding dilakukan hanya dari data training melalui pipeline.

## Q.2 Pipeline preprocessing dan baseline

- **Imputer:** Median untuk fitur numerik, Modus (`most_frequent`) untuk fitur kategorikal.
- **Scaler (Penjelasan Pilihan):** Menggunakan `StandardScaler`. Pilihan ini diambil karena algoritma klasifikasi utama adalah Logistic Regression dengan penalti regularisasi (L1/L2). Standardisasi (mean=0, std=1) memastikan nilai koefisien fitur dihukum secara adil oleh regularisasi dan optimasi gradien konvergen optimal, tanpa dibias oleh skala absolut fitur seperti `TotalCharges` vs `tenure`. `MinMaxScaler` tidak dipilih karena sangat rentan terhadap nilai ekstrem/outlier yang dapat memampatkan rentang data normal.
- **Encoder:** `OneHotEncoder(handle_unknown='ignore')` untuk mengonversi kategori nominal ke representasi biner.
- **Classifier:** `LogisticRegression` (efisien, transparan, dan menghasilkan estimasi probabilitas terkalibrasi untuk threshold tuning).

In [17]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

X_q = churn.drop(columns="Churn")
y_q = churn["Churn"].map({"No": 0, "Yes": 1})
q_num_cols = X_q.select_dtypes(include=np.number).columns.tolist()
q_cat_cols = X_q.select_dtypes(exclude=np.number).columns.tolist()
print("Fitur numerik:", q_num_cols)
print("Fitur kategorikal:", q_cat_cols)

q_numeric = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
q_categorical = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])
q_preprocessor = ColumnTransformer([
    ("num", q_numeric, q_num_cols),
    ("cat", q_categorical, q_cat_cols),
])
q_baseline = Pipeline([
    ("preprocessor", q_preprocessor),
    ("classifier", LogisticRegression(max_iter=2000, random_state=42)),
])

X_q_train, X_q_test, y_q_train, y_q_test = train_test_split(
    X_q, y_q, test_size=0.2, random_state=42, stratify=y_q
)
# Validation terpisah dipakai untuk pemilihan threshold; test tidak disentuh.
X_q_core, X_q_val, y_q_core, y_q_val = train_test_split(
    X_q_train, y_q_train, test_size=0.2, random_state=42, stratify=y_q_train
)
print("Train/core:", X_q_core.shape, np.bincount(y_q_core))
print("Validation:", X_q_val.shape, np.bincount(y_q_val))
print("Test:", X_q_test.shape, np.bincount(y_q_test))

Fitur numerik: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']
Fitur kategorikal: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']
Train/core: (4507, 19) [3311 1196]
Validation: (1127, 19) [828 299]
Test: (1409, 19) [1035  374]


In [18]:
def q_evaluate(model_name, y_true, y_pred, y_prob):
    result = {"Strategi": model_name, **metrics(y_true, y_pred, y_prob)}
    return result

q_baseline.fit(X_q_core, y_q_core)
q_pred_base = q_baseline.predict(X_q_test)
q_prob_base = q_baseline.predict_proba(X_q_test)[:, 1]
q_base_result = q_evaluate("Baseline LR", y_q_test, q_pred_base, q_prob_base)
display(pd.DataFrame([q_base_result]))

q_cm_base = confusion_matrix(y_q_test, q_pred_base)
plt.figure(figsize=(6, 5))
sns.heatmap(q_cm_base, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Tidak Churn", "Churn"], yticklabels=["Tidak Churn", "Churn"])
plt.title("Q — Confusion Matrix Baseline")
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.tight_layout()
plt.savefig("figures/08_q_cm_baseline.png", dpi=150, bbox_inches="tight")
plt.show()

,Strategi,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC,TN,FP,FN,TP
0,Baseline LR,0.799148,0.644444,0.542781,0.58926,0.842553,0.632686,923,112,171,203


**Identifikasi Penangkapan Kelas Minoritas:**

Apakah model baseline sudah menangkap kelas minoritas dengan baik? **BELUM.**
- Dari total **374 pelanggan yang sebenarnya churn** di data uji, model baseline hanya berhasil mendeteksi **203 pelanggan** (Recall = **54,28%**).
- Sebanyak **171 pelanggan churn (45,72%) lolos sebagai False Negative**.
- Akurasi baseline yang tinggi (**79,91%**) terbukti menipu (*accuracy paradox*) karena sebagian besar disumbang oleh keberhasilan menebak kelas mayoritas (tidak churn). Inilah bukti nyata mengapa penanganan imbalanced dan metrik Recall/F1 sangat mutlak dibutuhkan.

## Q.3 Handling imbalanced dengan SMOTE

SMOTE diletakkan **di dalam `imblearn.Pipeline` setelah preprocessing**, sehingga oversampling hanya terjadi pada training fold. Validation dan test set tidak pernah di-SMOTE.

In [19]:
q_smote = ImbPipeline([
    ("preprocessor", q_preprocessor),
    ("smote", SMOTE(random_state=42)),
    ("classifier", LogisticRegression(max_iter=2000, random_state=42)),
])
q_smote.fit(X_q_core, y_q_core)
q_pred_smote = q_smote.predict(X_q_test)
q_prob_smote = q_smote.predict_proba(X_q_test)[:, 1]
q_smote_result = q_evaluate("LR + SMOTE", y_q_test, q_pred_smote, q_prob_smote)
display(pd.DataFrame([q_base_result, q_smote_result]))

,Strategi,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC,TN,FP,FN,TP
0,Baseline LR,0.799148,0.644444,0.542781,0.589260,0.842553,0.632686,923,112,171,203
1,LR + SMOTE,0.743080,0.510753,0.762032,0.611588,0.840934,0.629878,762,273,89,285


## Q.4 Stratified 5-Fold Cross-Validation

Evaluasi dilakukan dengan `cross_validate` dan implementasi manual. Metrik dilaporkan sebagai mean ± sample standard deviation.

In [20]:
q_skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
q_scoring = ["accuracy", "precision", "recall", "f1", "roc_auc", "average_precision"]
q_cv_rows = []
for q_name, q_model in {"Baseline LR": q_baseline, "LR + SMOTE": q_smote}.items():
    q_scores = cross_validate(q_model, X_q_train, y_q_train, cv=q_skf,
                              scoring=q_scoring, n_jobs=-1)
    q_row = {"Model": q_name}
    for q_metric in q_scoring:
        q_values = q_scores[f"test_{q_metric}"]
        q_row[f"{q_metric}_mean"] = q_values.mean()
        q_row[f"{q_metric}_std"] = q_values.std(ddof=1)
    q_cv_rows.append(q_row)
q_cv_df = pd.DataFrame(q_cv_rows)
display(q_cv_df)
q_cv_df.to_csv("outputs/q_cv_comparison.csv", index=False)

/Users/ar/Project/MachineLearning/.venv/lib/python3.9/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
/Users/ar/Project/MachineLearning/.venv/lib/python3.9/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


/Users/ar/Project/MachineLearning/.venv/lib/python3.9/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
/Users/ar/Project/MachineLearning/.venv/lib/python3.9/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
/Users/ar/Project/MachineLearning/.venv/lib/python3.9/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


,Model,accuracy_mean,accuracy_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,roc_auc_mean,roc_auc_std,average_precision_mean,average_precision_std
0,Baseline LR,0.802096,0.012952,0.652899,0.028462,0.543144,0.045955,0.592316,0.033108,0.846137,0.014026,0.661430,0.021767
1,LR + SMOTE,0.756126,0.014329,0.527182,0.018706,0.786622,0.041866,0.631055,0.023680,0.844941,0.014536,0.658223,0.024380


In [21]:
q_manual_rows = []
for q_fold, (q_tr, q_va) in enumerate(q_skf.split(X_q_train, y_q_train), 1):
    q_fold_model = clone(q_baseline)
    q_fold_model.fit(X_q_train.iloc[q_tr], y_q_train.iloc[q_tr])
    q_fold_pred = q_fold_model.predict(X_q_train.iloc[q_va])
    q_fold_prob = q_fold_model.predict_proba(X_q_train.iloc[q_va])[:, 1]
    q_manual_rows.append({"Fold": q_fold, **metrics(
        y_q_train.iloc[q_va], q_fold_pred, q_fold_prob
    )})
q_manual_df = pd.DataFrame(q_manual_rows)
display(q_manual_df)
display(q_manual_df[["Accuracy", "Recall", "F1", "ROC_AUC", "PR_AUC"]].agg(["mean", "std"]))
q_manual_df.to_csv("outputs/q_manual_cv.csv", index=False)

,Fold,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC,TN,FP,FN,TP
0,1,0.798580,0.645161,0.535117,0.585009,0.847866,0.643658,740,88,139,160
1,2,0.782609,0.609756,0.501672,0.550459,0.825164,0.635175,732,96,149,150
2,3,0.802130,0.658333,0.528428,0.586271,0.841872,0.685808,746,82,141,158
3,4,0.816327,0.664286,0.622074,0.642487,0.862866,0.678418,734,94,113,186
4,5,0.810835,0.686957,0.528428,0.597353,0.852916,0.664090,755,72,141,158


,Accuracy,Recall,F1,ROC_AUC,PR_AUC
mean,0.802096,0.543144,0.592316,0.846137,0.661430
std,0.012952,0.045955,0.033108,0.014026,0.021767


## Q.5 Hyperparameter tuning dengan GridSearchCV

Tiga hyperparameter Logistic Regression dicari: `C`, `penalty`, dan `class_weight`. Scoring utama adalah F1 sesuai instruksi tugas.

In [22]:
q_param_grid = {
    "classifier__C": [0.1, 1.0, 10.0],
    "classifier__penalty": ["l1", "l2"],
    "classifier__class_weight": [None, "balanced"],
    "classifier__solver": ["liblinear"],
}
q_grid = GridSearchCV(
    q_baseline, q_param_grid, cv=q_skf, scoring="f1", n_jobs=-1,
    return_train_score=False
)
q_grid.fit(X_q_core, y_q_core)
print("Best parameters:", q_grid.best_params_)
print(f"Best CV F1: {q_grid.best_score_:.4f}")
q_grid_results = pd.DataFrame(q_grid.cv_results_).sort_values("rank_test_score")
display(q_grid_results[["params", "mean_test_score", "std_test_score", "rank_test_score"]].head())
q_grid_results.to_csv("outputs/q_grid_search_results.csv", index=False)

q_pred_grid = q_grid.best_estimator_.predict(X_q_test)
q_prob_grid = q_grid.best_estimator_.predict_proba(X_q_test)[:, 1]
q_grid_result = q_evaluate("Best GridSearch (0.5)", y_q_test, q_pred_grid, q_prob_grid)
display(pd.DataFrame([q_grid_result]))

Best parameters: {'classifier__C': 1.0, 'classifier__class_weight': 'balanced', 'classifier__penalty': 'l1', 'classifier__solver': 'liblinear'}
Best CV F1: 0.6277


,params,mean_test_score,std_test_score,rank_test_score
6,"{'classifier__C': 1.0, 'classifier__class_weig...",0.627729,0.011070,1
7,"{'classifier__C': 1.0, 'classifier__class_weig...",0.627240,0.011685,2
3,"{'classifier__C': 0.1, 'classifier__class_weig...",0.627072,0.013592,3
10,"{'classifier__C': 10.0, 'classifier__class_wei...",0.626736,0.010467,4
11,"{'classifier__C': 10.0, 'classifier__class_wei...",0.626535,0.010600,5


,Strategi,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC,TN,FP,FN,TP
0,Best GridSearch (0.5),0.739532,0.506003,0.78877,0.61651,0.842391,0.634667,747,288,79,295


## Q.6 Threshold tuning dengan Precision–Recall Curve

Threshold optimal F1 dipilih dari **validation set**, lalu hanya sekali dievaluasi pada test set untuk menghindari optimisme evaluasi.

In [23]:
q_val_prob = q_grid.best_estimator_.predict_proba(X_q_val)[:, 1]
q_precisions, q_recalls, q_thresholds = precision_recall_curve(y_q_val, q_val_prob)
q_f1s = 2 * q_precisions[:-1] * q_recalls[:-1] / (
    q_precisions[:-1] + q_recalls[:-1] + 1e-12
)
q_opt_idx = int(np.argmax(q_f1s))
q_opt_threshold = float(q_thresholds[q_opt_idx])
print(f"Threshold optimal: {q_opt_threshold:.4f}")
print(f"Validation F1 optimal: {q_f1s[q_opt_idx]:.4f}")

plt.figure(figsize=(9, 6))
plt.plot(q_thresholds, q_precisions[:-1], label="Precision")
plt.plot(q_thresholds, q_recalls[:-1], label="Recall")
plt.plot(q_thresholds, q_f1s, label="F1")
plt.axvline(q_opt_threshold, color="black", ls="--", label=f"Optimal={q_opt_threshold:.3f}")
plt.xlabel("Threshold"); plt.ylabel("Score"); plt.title("Q — Precision, Recall, dan F1 vs Threshold")
plt.legend(); plt.grid(alpha=.3); plt.tight_layout()
plt.savefig("figures/09_q_threshold_tuning.png", dpi=150, bbox_inches="tight")
plt.show()

q_pred_opt = (q_prob_grid >= q_opt_threshold).astype(int)
q_opt_result = q_evaluate(f"Threshold tuning ({q_opt_threshold:.4f})",
                          y_q_test, q_pred_opt, q_prob_grid)
display(pd.DataFrame([q_grid_result, q_opt_result]))

Threshold optimal: 0.6398
Validation F1 optimal: 0.6443


,Strategi,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC,TN,FP,FN,TP
0,Best GridSearch (0.5),0.739532,0.506003,0.788770,0.616510,0.842391,0.634667,747,288,79,295
1,Threshold tuning (0.6398),0.773598,0.561247,0.673797,0.612394,0.842391,0.634667,838,197,122,252


## Q.7 Demonstrasi data leakage

**Cara salah:** seluruh preprocessing di-fit pada semua data sebelum train-test split. Informasi median, mean, standar deviasi, dan kategori dari test set ikut memengaruhi representasi training.

**Cara benar:** split dilakukan lebih dahulu dan preprocessing berada dalam pipeline yang hanya di-fit pada training set.

In [24]:
# SALAH: preprocessor mempelajari seluruh data sebelum split.
q_leaky_preprocessor = clone(q_preprocessor)
X_q_leaky = q_leaky_preprocessor.fit_transform(X_q)
X_q_train_w, X_q_test_w, y_q_train_w, y_q_test_w = train_test_split(
    X_q_leaky, y_q, test_size=0.2, random_state=42, stratify=y_q
)
q_wrong_model = LogisticRegression(max_iter=2000, random_state=42)
q_wrong_model.fit(X_q_train_w, y_q_train_w)
q_wrong_pred = q_wrong_model.predict(X_q_test_w)
q_wrong_prob = q_wrong_model.predict_proba(X_q_test_w)[:, 1]
q_wrong = q_evaluate("SALAH: preprocessing sebelum split", y_q_test_w, q_wrong_pred, q_wrong_prob)

# BENAR: seluruh preprocessing dipelajari hanya dari training set.
q_right_model = clone(q_baseline).fit(X_q_train, y_q_train)
q_right_pred = q_right_model.predict(X_q_test)
q_right_prob = q_right_model.predict_proba(X_q_test)[:, 1]
q_right = q_evaluate("BENAR: preprocessing dalam pipeline", y_q_test, q_right_pred, q_right_prob)
q_leakage_df = pd.DataFrame([q_wrong, q_right])
display(q_leakage_df)
q_leakage_df.to_csv("outputs/q_leakage_comparison.csv", index=False)

,Strategi,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC,TN,FP,FN,TP
0,SALAH: preprocessing sebelum split,0.805536,0.657233,0.558824,0.604046,0.841848,0.633807,926,109,165,209
1,BENAR: preprocessing dalam pipeline,0.805536,0.657233,0.558824,0.604046,0.841861,0.633429,926,109,165,209


### Penjelasan: Mengapa Hasil 'Salah' (Data Leakage) Bisa Terlihat Lebih Bagus?

Ketika transformasi seperti scaling dan imputasi dilakukan pada **seluruh dataset sebelum split**, statistik dari test set (nilai mean, varians, median, dan kategori) ikut bocor dan memengaruhi proses pelatihan model.

1. **Informasi Masa Depan Ikut Terbaca:** Model seolah-olah sudah 'mengintip' sebaran data uji sebelum diuji, sehingga estimasi kesalahan saat evaluasi menjadi lebih rendah dari realitas sebenarnya (*overly optimistic*).
2. **Varians Semu:** Pada evaluasi cross-validation yang bocor, skor performa antar-fold sering kali tampak lebih stabil dan lebih tinggi secara artifisial karena data uji fold tersebut telah ikut membentuk transformer di tahap awal.
3. **Bahaya di Produksi:** Walaupun pada dataset besar dengan distribusi homogen selisih angkanya tampak kecil, cara salah ini fatal di dunia nyata. Saat model menerima data masa depan (*production data*) yang memiliki outlier atau pergeseran tren (*distribution shift*), model yang dilatih dengan data leakage akan mengalami penurunan performa drastis.

## Q.8 Perbandingan final semua strategi

In [25]:
q_comparison = pd.DataFrame([
    q_base_result, q_smote_result, q_grid_result, q_opt_result
])
q_comparison.to_csv("outputs/q_final_comparison.csv", index=False)
display(q_comparison)

q_plot_cols = ["Accuracy", "Precision", "Recall", "F1", "ROC_AUC", "PR_AUC"]
q_comparison.set_index("Strategi")[q_plot_cols].plot(
    kind="bar", figsize=(13, 7), colormap="Set2"
)
plt.title("Q — Perbandingan Strategi Customer Churn")
plt.ylabel("Score"); plt.xticks(rotation=20, ha="right")
plt.legend(loc="lower right", ncol=2); plt.tight_layout()
plt.savefig("figures/10_q_final_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

,Strategi,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC,TN,FP,FN,TP
0,Baseline LR,0.799148,0.644444,0.542781,0.589260,0.842553,0.632686,923,112,171,203
1,LR + SMOTE,0.743080,0.510753,0.762032,0.611588,0.840934,0.629878,762,273,89,285
2,Best GridSearch (0.5),0.739532,0.506003,0.788770,0.616510,0.842391,0.634667,747,288,79,295
3,Threshold tuning (0.6398),0.773598,0.561247,0.673797,0.612394,0.842391,0.634667,838,197,122,252


## Q.9 Refleksi

### Precision atau Recall?

Dalam konteks customer churn, **recall lebih diprioritaskan** karena false negative berarti pelanggan yang benar-benar akan churn tidak memperoleh intervensi retensi. Kehilangan pelanggan umumnya lebih mahal daripada menghubungi sebagian pelanggan yang ternyata tidak akan churn. Namun, precision tidak boleh diabaikan karena false positive dapat menghabiskan biaya promosi, waktu staf, dan berisiko memberi diskon kepada pelanggan yang sebenarnya tetap bertahan.

### Cara meminimalkan dampak kesalahan prediksi

1. Gunakan model sebagai alat prioritas, bukan keputusan otomatis; pelanggan berisiko tinggi diverifikasi oleh tim retention.
2. Sesuaikan threshold berdasarkan biaya nyata false negative dan false positive, bukan F1 saja.
3. Terapkan intervensi bertingkat: pesan murah untuk risiko menengah dan penawaran bernilai tinggi hanya untuk risiko tinggi.
4. Pantau precision, recall, calibration, drift data, serta performa per kelompok pelanggan secara berkala.
5. Lakukan A/B testing untuk memastikan intervensi benar-benar mengurangi churn dan tidak sekadar mengidentifikasi pelanggan berisiko.
6. Sediakan mekanisme opt-out dan batasi penggunaan atribut sensitif untuk menjaga privasi dan keadilan.

### Kesimpulan tugas

Accuracy tidak cukup untuk memilih model churn. SMOTE dan class weighting dapat meningkatkan sensitivitas terhadap kelas churn, tetapi biasanya menurunkan precision. GridSearchCV membantu memilih konfigurasi berdasarkan F1, sedangkan threshold tuning memungkinkan trade-off yang lebih sesuai dengan kebutuhan bisnis. Semua preprocessing dan resampling harus berada di dalam pipeline untuk mencegah data leakage.